# TP — Support de code : comprendre les principes de l'apprentissage automatique

Notebook compagnon de la séance *« Comprendre les principes de l'apprentissage automatique »* (INSA Strasbourg).
Il reprend les démonstrations du support, **dans le même ordre**, pour que vous puissiez les faire tourner vous-même et répondre aux questions des participants.

Conçu pour tourner directement sur **Google Colab**, sans rien installer.

Les parties « **Exercice** » vont un peu plus loin que le support — elles sont volontairement **moins commentées**.

## Sommaire

1. Régression : `fit()` apprend les paramètres
2. Classification : frontière de décision
3. Généraliser : sous-apprentissage / sur-apprentissage
4. Choisir un hyperparamètre sans tricher : validation croisée
5. Évaluer : au-delà de l'exactitude
6. Non supervisé : K-means
7. Non supervisé : détection d'anomalies
8. Bonus — pour aller plus loin (annexes du support)

## 0. Préparatifs

In [ ]:
import warnings; warnings.filterwarnings("ignore")  # quelques warnings numériques inoffensifs, sans intérêt ici
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from numpy.exceptions import RankWarning  # numpy >= 2.0
except ImportError:
    from numpy import RankWarning              # numpy < 2.0
warnings.filterwarnings("ignore", category=RankWarning)

plt.rcParams.update({"figure.figsize": (8, 4.5), "font.size": 12,
                      "axes.spines.top": False, "axes.spines.right": False})

BLEU, ORANGE, VERT, ROUGE = "#0072B2", "#E69F00", "#009E73", "#D55E00"

## 1. Régression : `fit()` apprend les paramètres du modèle

Jeu de données *Diabetes* (442 patients). Une seule variable d'entrée : l'IMC.
La cible : évolution de la maladie un an après l'examen initial.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

X, y = load_diabetes(return_X_y=True, as_frame=True, scaled=False)
X = X[["bmi"]]  # une seule variable : l'IMC

# train_test_split : sépare X, y en un jeu d'entraînement (80 %) et un jeu de test (20 %),
# tirés aléatoirement (random_state fixe le tirage, pour un résultat reproductible)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"{len(X_train)} patients d'entraînement, {len(X_test)} patients de test")

Nuage de points **avant tout apprentissage** — rien n'a encore été ajusté.

In [ ]:
plt.scatter(X_train, y_train, color=BLEU, alpha=0.4, label="patients d'entraînement")
plt.xlabel("Indice de masse corporelle")
plt.ylabel("Évolution de la maladie à 1 an")
plt.legend()
plt.show()

`fit()` estime les deux paramètres de la droite (pente et ordonnée à l'origine).

In [ ]:
from sklearn.linear_model import LinearRegression

modele = LinearRegression()
modele.fit(X_train, y_train)  # apprentissage : trouve a et b qui minimisent l'erreur sur le train

print("pente (a)               :", round(modele.coef_[0], 2))
print("ordonnée à l'origine (b) :", round(modele.intercept_, 2))

In [ ]:
xs = np.linspace(X_train["bmi"].min(), X_train["bmi"].max(), 100).reshape(-1, 1)

plt.scatter(X_train, y_train, color=BLEU, alpha=0.25, label="patients d'entraînement")
plt.plot(xs, modele.predict(xs), color=ROUGE, linewidth=2, label="droite ajustée")  # predict() : applique le modèle appris à de nouvelles valeurs
plt.xlabel("Indice de masse corporelle")
plt.ylabel("Évolution de la maladie à 1 an")
plt.legend()
plt.show()

`predict()` sur 3 patients **jamais vus** (jeu de test).

In [ ]:
X_new = X_test.head(3)
y_new = y_test.head(3)
y_hat = modele.predict(X_new)  # predict() sur des patients du jeu de test : jamais vus pendant fit()

tableau = pd.DataFrame({
    "IMC": X_new["bmi"].round(1).values,
    "prédit": y_hat.round(0).astype(int),
    "observé": y_new.astype(int).values,
})
tableau["écart"] = tableau["prédit"] - tableau["observé"]
tableau

> À IMC comparable, les patients réels varient beaucoup : une seule variable ne suffit pas à bien expliquer la cible.

### Exercice 1 — aller plus loin

Reprenez les mêmes étapes en utilisant **toutes** les variables du dataset (pas seulement `bmi`).
Le score R² (`modele.score(X_test, y_test)`) s'améliore-t-il ?

In [ ]:
X_full, y_full = load_diabetes(return_X_y=True, as_frame=True, scaled=False)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=0)

modele_full = LinearRegression().fit(X_train_f, y_train_f)

# score() : calcule le R² sur des données de test (1 = parfait, 0 = pas mieux qu'une moyenne)
print("R² avec 1 seule variable (bmi) :", round(modele.score(X_test, y_test), 3))
print("R² avec toutes les variables    :", round(modele_full.score(X_test_f, y_test_f), 3))

## 2. Classification : le modèle apprend une frontière de décision

Jeu de données *Iris* (150 fleurs, 3 espèces). Deux variables : longueur et largeur du pétale.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from matplotlib.colors import ListedColormap

iris = load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]]
y = iris.target

modele_iris = LogisticRegression()
modele_iris.fit(X, y)  # apprend une frontière de décision entre les 3 espèces

CMAP = ListedColormap([BLEU, ORANGE, VERT])
FOND = ListedColormap(["#E3EFF7", "#FCF1DE", "#E0F2EB"])

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X.iloc[:, 0].min() - 0.3, X.iloc[:, 0].max() + 0.3, 300),
    np.linspace(X.iloc[:, 1].min() - 0.3, X.iloc[:, 1].max() + 0.3, 300),
)
zz = modele_iris.predict(  # predict() sur chaque point de la grille, pour colorer les zones
    pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X.columns)
).reshape(xx.shape)

plt.contourf(xx, yy, zz, cmap=FOND, alpha=0.9)
plt.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y, cmap=CMAP, edgecolor="white", s=40)
plt.xlabel("Longueur du pétale (cm)")
plt.ylabel("Largeur du pétale (cm)")
plt.show()

On compte les désaccords entre prédiction et espèce réelle, **sur les données d'entraînement elles-mêmes**.

In [ ]:
y_pred = modele_iris.predict(X)  # predict() ici sur les données d'ENTRAÎNEMENT elles-mêmes
print("désaccords :", int((y_pred != y).sum()), "/", len(y))

> 145/150 correspondent... mais ces 150 observations ont aussi servi à **entraîner** le modèle : ce chiffre ne dit rien
> de sa capacité à généraliser (voir partie 3).

Même tâche, un autre modèle : un arbre de décision.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

arbre = DecisionTreeClassifier(max_depth=2, random_state=0)  # max_depth=2 : 2 questions maximum
arbre.fit(X, y)

plt.figure(figsize=(10, 4.6))
plot_tree(arbre, feature_names=list(X.columns), class_names=list(iris.target_names),  # dessine l'arbre appris
          filled=True, fontsize=10, rounded=True)
plt.show()

### Exercice 2 — aller plus loin

Comparez à un `KNeighborsClassifier` (k plus proches voisins, cf. annexe An3 du support).
Testez plusieurs valeurs de `k` : que se passe-t-il pour `k=1` ? pour un `k` très grand ?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in [1, 5, 30]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)  # classe selon la majorité des k plus proches voisins
    print(f"k={k:>2} -> exactitude sur les données d'entraînement : {knn.score(X, y):.3f}")

# À vous : tracez la frontière de décision pour k=1 et k=30 (même code que ci-dessus)
# et observez comment elle se déforme.

## 3. Généraliser : sous-apprentissage vs sur-apprentissage

Même principe que la comparaison des trois modèles du support (degré 1 / 3 / 9),
sur un jeu de données simulé à une seule variable.

In [ ]:
from numpy.polynomial import Polynomial
from sklearn.metrics import mean_squared_error

rng = np.random.RandomState(76)

def f_vraie(x):
    return np.cos(1.5 * np.pi * x)

n = 22
s_X = np.sort(rng.rand(n))
s_y = f_vraie(s_X) + rng.randn(n) * 0.10

s_X_tr, s_X_te, s_y_tr, s_y_te = train_test_split(
    s_X, s_y, test_size=1 - 10 / n, random_state=76)

DEGRES = [1, 9, 3]  # trop simple / trop complexe / juste
xx_p = np.linspace(0, 1, 400)
s_modeles = [Polynomial.fit(s_X_tr, s_y_tr, d) for d in DEGRES]  # Polynomial.fit : ajuste un polynôme de degré d aux points d'entraînement

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, d, m in zip(axes, DEGRES, s_modeles):
    ax.plot(xx_p, f_vraie(xx_p), "--", color="#8A8A8A", label="fonction réelle")
    ax.plot(xx_p, m(xx_p), color="black", lw=2.5, label="modèle appris")
    ax.scatter(s_X_tr, s_y_tr, color=BLEU, label="points vus")
    ax.scatter(s_X_te, s_y_te, marker="X", color=ROUGE, label="points jamais vus")
    ax.set_title(f"degré {d}")
axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

for d, m in zip(DEGRES, s_modeles):
    print(f"degré {d:2d} : erreur entraînement {mean_squared_error(s_y_tr, m(s_X_tr)):.2e} "
          f"| erreur données jamais vues {mean_squared_error(s_y_te, m(s_X_te)):.4f}")

> Le modèle « degré 9 » a l'erreur d'entraînement la plus faible des trois, et la pire en généralisation.

### Exercice 3 — aller plus loin

Faites varier le degré du polynôme de 1 à 15 et tracez l'erreur d'entraînement et l'erreur de test
en fonction du degré (une « courbe de validation »). À partir de quel degré l'écart se creuse-t-il ?

In [ ]:
degres_test = range(1, 16)
err_tr, err_te = [], []
for d in degres_test:
    m = Polynomial.fit(s_X_tr, s_y_tr, d)
    err_tr.append(mean_squared_error(s_y_tr, m(s_X_tr)))
    err_te.append(mean_squared_error(s_y_te, m(s_X_te)))

plt.plot(degres_test, err_tr, "o-", color=BLEU, label="erreur entraînement")
plt.plot(degres_test, err_te, "o-", color=ROUGE, label="erreur test")
plt.yscale("log")
plt.xlabel("degré du polynôme")
plt.ylabel("erreur quadratique moyenne")
plt.legend()
plt.show()

## 4. Choisir un hyperparamètre sans tricher : validation croisée

Comparer plusieurs degrés directement sur le jeu de test, c'est déjà « tricher » : le test cesse
d'être neutre. La validation croisée permet de comparer les degrés en utilisant uniquement le
jeu d'entraînement.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

for d in [1, 3, 9]:
    pipe = make_pipeline(PolynomialFeatures(d), LinearRegression())  # make_pipeline : chaîne les étapes (transforme puis apprend)
    # cross_val_score : découpe le train en 5 plis, entraîne et évalue 5 fois, sans jamais toucher au test
    scores = cross_val_score(pipe, s_X_tr.reshape(-1, 1), s_y_tr,
                              cv=5, scoring="neg_mean_squared_error")
    print(f"degré {d} : erreur moyenne (5-fold) = {-scores.mean():.4f}")

### Exercice 4

Changez `cv=5` en `cv=3` ou `cv=10`. Le classement des degrés change-t-il ?

In [ ]:
for cv_k in [3, 10]:
    print(f"--- cv={cv_k} ---")
    for d in [1, 3, 9]:
        pipe = make_pipeline(PolynomialFeatures(d), LinearRegression())
        scores = cross_val_score(pipe, s_X_tr.reshape(-1, 1), s_y_tr,
                                  cv=cv_k, scoring="neg_mean_squared_error")
        print(f"  degré {d} : erreur moyenne = {-scores.mean():.4f}")

## 5. Évaluer : au-delà de l'exactitude

Contrôle qualité (cf. support) : 10 000 pièces dont 50 défectueuses.
Un modèle en détecte 40 et en signale 100 au total.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# jeu d'étiquettes reconstruit pour retomber exactement sur les chiffres du support
y_true = np.array([1] * 50 + [0] * 9950)
y_pred = np.array([1] * 40 + [0] * 10 + [1] * 60 + [0] * 9890)

print(confusion_matrix(y_true, y_pred))               # confusion_matrix : les 4 cas (VP, FP, VN, FN) sous forme de tableau
print("rappel    :", round(recall_score(y_true, y_pred), 2))       # recall_score    : parmi les vrais défauts, combien détectés ?
print("précision :", round(precision_score(y_true, y_pred), 2))    # precision_score : parmi les défauts signalés, combien étaient vrais ?
print("F1        :", round(f1_score(y_true, y_pred), 2))           # f1_score        : moyenne harmonique de rappel et précision

### Exercice 5

Calculez ces mêmes métriques sur le modèle Iris de la partie 2, avec un vrai split train/test
cette fois (pas les 150 observations d'entraînement), via `classification_report`.

In [ ]:
from sklearn.metrics import classification_report

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
modele_eval = LogisticRegression().fit(X_tr_i, y_tr_i)
# classification_report : rappel/précision/F1 pour chaque classe, calculés d'un coup
print(classification_report(y_te_i, modele_eval.predict(X_te_i), target_names=iris.target_names))

## 6. Non supervisé : K-means

Mêmes iris, mêmes 2 variables, mais **sans** la colonne espèce.

In [ ]:
from sklearn.cluster import KMeans

X_nu = iris.data[["petal length (cm)", "petal width (cm)"]]  # y n'est pas utilisé

kmeans3 = KMeans(n_clusters=3, n_init=10, random_state=0)  # K=3 est un hyperparamètre
labels3 = kmeans3.fit_predict(X_nu)  # fit_predict : apprend les 3 centres ET renvoie le groupe de chaque point

plt.scatter(X_nu.iloc[:, 0], X_nu.iloc[:, 1], c=labels3, cmap=CMAP, alpha=0.7, s=40)
plt.scatter(kmeans3.cluster_centers_[:, 0], kmeans3.cluster_centers_[:, 1],
            color="black", marker="X", s=160, label="centres appris")
plt.xlabel("Longueur du pétale (cm)")
plt.ylabel("Largeur du pétale (cm)")
plt.legend()
plt.show()

Et si on demande **2** groupes au lieu de 3 ?

In [ ]:
CMAP2 = ListedColormap([BLEU, ORANGE])

kmeans2 = KMeans(n_clusters=2, n_init=10, random_state=0)
labels2 = kmeans2.fit_predict(X_nu)

plt.scatter(X_nu.iloc[:, 0], X_nu.iloc[:, 1], c=labels2, cmap=CMAP2, alpha=0.7, s=40)
plt.scatter(kmeans2.cluster_centers_[:, 0], kmeans2.cluster_centers_[:, 1],
            color="black", marker="X", s=160, label="centres appris")
plt.xlabel("Longueur du pétale (cm)")
plt.ylabel("Largeur du pétale (cm)")
plt.legend()
plt.show()

> Un cluster n'est pas une classe : la correspondance avec les vraies espèces n'est établie qu'**après coup**.

### Exercice 6 — aller plus loin

Testez `n_clusters` de 1 à 8 et tracez l'inertie (`kmeans.inertia_`) en fonction de K :
c'est la **méthode du coude** (cf. annexe An5 du support).

In [ ]:
inerties = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_nu)
    inerties.append(km.inertia_)  # inertia_ : somme des distances au carré entre chaque point et son centre

plt.plot(range(1, 9), inerties, "o-", color=BLEU)
plt.xlabel("K")
plt.ylabel("inertie")
plt.show()

## 7. Non supervisé : détection d'anomalies

On génère un nuage compact et on y injecte un point clairement aberrant.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest

Xa, _ = make_blobs(n_samples=150, centers=1, cluster_std=0.5, random_state=42)
Xa[-1, :] = np.array((5, 5))  # on injecte un point aberrant

detecteur = IsolationForest(contamination="auto", random_state=0).fit(Xa)  # apprend ce qu'est un point "normal" à partir des données
anomalies = detecteur.predict(Xa)  # -1 = anomalie, 1 = normal

plt.scatter(Xa[:, 0], Xa[:, 1], c=np.where(anomalies == -1, ROUGE, BLEU), alpha=0.7)
plt.show()

print("points signalés :", int((anomalies == -1).sum()))

### Exercice 7

Faites varier `contamination` (ex. 0.01, 0.05, 0.2). Combien de points sont signalés à chaque fois ?

In [ ]:
for c in [0.01, 0.05, 0.2]:
    d = IsolationForest(contamination=c, random_state=0).fit(Xa)
    n_signales = int((d.predict(Xa) == -1).sum())
    print(f"contamination={c:<5} -> {n_signales} points signalés")

## 8. Bonus — pour aller plus loin (annexes du support)

### Réduction de dimension : ACP (annexe An12)

On repasse aux 4 variables du dataset Iris (au lieu de 2), et on les résume en 2 composantes.

In [ ]:
from sklearn.decomposition import PCA

X4 = iris.data  # les 4 variables cette fois
acp = PCA(n_components=2).fit(X4)  # cherche les 2 axes qui résument le mieux les 4 variables
X_reduit = acp.transform(X4)       # transform : projette chaque observation sur ces 2 axes

print("variance expliquée par composante :", acp.explained_variance_ratio_.round(3))

plt.scatter(X_reduit[:, 0], X_reduit[:, 1], c=iris.target, cmap=CMAP, alpha=0.7)
plt.xlabel("composante 1")
plt.ylabel("composante 2")
plt.show()

### Régularisation : Ridge et Lasso (annexe An8)

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0).fit(X_train_f, y_train_f)  # comme une régression linéaire, mais pénalise les gros coefficients
lasso = Lasso(alpha=1.0).fit(X_train_f, y_train_f)  # idem, mais peut ramener certains coefficients exactement à 0

coeffs = pd.DataFrame({
    "variable": X_train_f.columns,
    "OLS (sans pénalité)": modele_full.coef_.round(1),
    "Ridge": ridge.coef_.round(1),
    "Lasso": lasso.coef_.round(1),
})
coeffs

## Pour continuer

- **scikit-learn** : `scikit-learn.org` — la documentation contient un exemple exécutable pour presque chaque fonction utilisée ici.
- **pandas** : `pandas.pydata.org`
- Tous les modèles vus ici suivent le même schéma : `modele.fit(X, y)` puis `modele.predict(X_nouveau)`.
  Les séances suivantes (réseaux de neurones) réutilisent exactement ce même schéma.